In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, avg

spark = SparkSession.builder \
    .appName("Popularity Recommendation") \
    .getOrCreate()

data = [
    (1, 101, 4.5),
    (1, 102, 3.0),
    (2, 101, 4.0),
    (2, 104, 4.5),
    (3, 101, 5.0),
    (3, 103, 4.0),
    (4, 104, 5.0),
    (5, 101, 4.5)
]

columns = ["user_id", "movie_id", "rating"]

df = spark.createDataFrame(data, columns)
df.show()


+-------+--------+------+
|user_id|movie_id|rating|
+-------+--------+------+
|      1|     101|   4.5|
|      1|     102|   3.0|
|      2|     101|   4.0|
|      2|     104|   4.5|
|      3|     101|   5.0|
|      3|     103|   4.0|
|      4|     104|   5.0|
|      5|     101|   4.5|
+-------+--------+------+



In [ ]:
#Find Most Popular Movies
popularity_df = df.groupBy("movie_id") \
    .agg(
        count("rating").alias("total_views"),
        avg("rating").alias("avg_rating")
    ) \
    .orderBy("total_views", "avg_rating", ascending=False)

popularity_df.show()


+--------+-----------+----------+
|movie_id|total_views|avg_rating|
+--------+-----------+----------+
|     101|          4|       4.5|
|     104|          2|      4.75|
|     103|          1|       4.0|
|     102|          1|       3.0|
+--------+-----------+----------+



In [ ]:
movies = [
    (101, "Stranger Things"),
    (102, "Money Heist"),
    (103, "Dark"),
    (104, "Extraction")
]

movie_df = spark.createDataFrame(movies, ["movie_id", "movie_name"])

final_df = popularity_df.join(movie_df, "movie_id")
final_df.show()


+--------+-----------+----------+---------------+
|movie_id|total_views|avg_rating|     movie_name|
+--------+-----------+----------+---------------+
|     101|          4|       4.5|Stranger Things|
|     102|          1|       3.0|    Money Heist|
|     103|          1|       4.0|           Dark|
|     104|          2|      4.75|     Extraction|
+--------+-----------+----------+---------------+



In [ ]:
from pyspark.sql.functions import avg

highly_rated = df.groupBy("movie_id") \
    .agg(avg("rating").alias("avg_rating")) \
    .orderBy("avg_rating", ascending=False)

highly_rated.show()


+--------+----------+
|movie_id|avg_rating|
+--------+----------+
|     104|      4.75|
|     101|       4.5|
|     103|       4.0|
|     102|       3.0|
+--------+----------+



In [ ]:

#Popular + High Rating
from pyspark.sql.functions import avg, count

recommended = df.groupBy("movie_id") \
    .agg(
        count("rating").alias("total_views"),
        avg("rating").alias("avg_rating")
    ) \
    .filter("total_views >= 3") \
    .orderBy("avg_rating", "total_views", ascending=False)

recommended.show()


+--------+-----------+----------+
|movie_id|total_views|avg_rating|
+--------+-----------+----------+
|     101|          4|       4.5|
+--------+-----------+----------+



In [ ]:
from pyspark.sql.functions import avg, count

top_movies = df.groupBy("movie_id") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("total_views")
    ) \
    .orderBy("avg_rating", "total_views", ascending=False)

top_movies.show()


+--------+----------+-----------+
|movie_id|avg_rating|total_views|
+--------+----------+-----------+
|     104|      4.75|          2|
|     101|       4.5|          4|
|     103|       4.0|          1|
|     102|       3.0|          1|
+--------+----------+-----------+



In [ ]:
final_movies = top_movies.join(movie_df, "movie_id")
final_movies.show()


+--------+----------+-----------+---------------+
|movie_id|avg_rating|total_views|     movie_name|
+--------+----------+-----------+---------------+
|     101|       4.5|          4|Stranger Things|
|     102|       3.0|          1|    Money Heist|
|     103|       4.0|          1|           Dark|
|     104|      4.75|          2|     Extraction|
+--------+----------+-----------+---------------+



In [ ]:
final_movies.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("top_movies_powerbi")


In [ ]:
pip install kaggle

In [ ]:
pip install kagglehub pandas

In [ ]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")

print("Dataset downloaded to:", path)

# Read CSV
csv_file = os.path.join(path, "netflix_titles.csv")

df = pd.read_csv(csv_file)

print(df.head())

Using Colab cache for faster access to the 'netflix-shows' dataset.
Dataset downloaded to: /kaggle/input/netflix-shows
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021   

In [ ]:
import pandas as pd

df = pd.read_csv(r"/kaggle/input/netflix-shows")
df.head()

IsADirectoryError: [Errno 21] Is a directory: '/kaggle/input/netflix-shows'

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/amazon-prime-movies-and-tv-shows")

print("Path to dataset files:", path)

amazon_prime_titles.csv

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/amazon-prime-movies-and-tv-shows")

print("Path to dataset files:", path)

csv_file = os.path.join(path,"amazon_prime_titles.csv")

df = pd.read_csv(csv_file)

print(df.head())

Using Colab cache for faster access to the 'amazon-prime-movies-and-tv-shows' dataset.
Path to dataset files: /kaggle/input/amazon-prime-movies-and-tv-shows
  show_id   type                 title        director  \
0      s1  Movie   The Grand Seduction    Don McKellar   
1      s2  Movie  Take Care Good Night    Girish Joshi   
2      s3  Movie  Secrets of Deception     Josh Webber   
3      s4  Movie    Pink: Staying True  Sonia Anderson   
4      s5  Movie         Monster Maker    Giles Foster   

                                                cast         country  \
0     Brendan Gleeson, Taylor Kitsch, Gordon Pinsent          Canada   
1   Mahesh Manjrekar, Abhay Mahajan, Sachin Khedekar           India   
2  Tom Sizemore, Lorenzo Lamas, Robert LaSardo, R...   United States   
3  Interviews with: Pink, Adele, Beyoncé, Britney...   United States   
4  Harry Dean Stanton, Kieran O'Brien, George Cos...  United Kingdom   

       date_added  release_year rating duration              